# 05 - Bias Drift Monitoring

This notebook tracks how each outlet's predicted bias label distribution changes over time.
The idea is simple: if BBC is consistently classified as 60% left / 40% centre in November 2025,
but shifts to 20% left / 80% centre by March 2026, something has changed - either the outlet's
editorial direction, the topics being covered, or both.

**Data sources combined here**:
- GDELT historical articles (Nov 2025 - Apr 2026) - sampled monthly, 1-7 articles per outlet per month
- Live RSS + NewsAPI articles (May 2026, single ingestion run) - 10-46 articles per outlet

**Design choice - monthly aggregation**: the original plan called for 7-day rolling averages.
That requires daily data density. GDELT provides 2-5 articles per outlet per month, so weekly
windows would be mostly empty. Monthly proportions are the right granularity for this dataset.
This is documented honestly in the observations section.

**Outlets monitored**: BBC, NPR, Fox News, The Guardian
These are the four outlets that have GDELT historical data. All others only have May 2026 data
and cannot be shown on a time axis.

## 1. Setup

In [9]:
import sys
import os

# Add project root to sys.path so I can import from src/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.drift import (
    load_drift_data,
    monthly_bias_proportions,
    compute_baselines,
    detect_drift_events,
    DRIFT_OUTLETS,
)

# Load all articles for the four monitored outlets
df = load_drift_data()

print(f"Articles loaded: {len(df)}")
print()
print(df.groupby(["outlet", "source"])["bias_label"].count().rename("articles").to_string())

Articles loaded: 183

outlet        source 
bbc           gdelt      23
              newsapi     1
              rss        37
fox_news      gdelt      13
              newsapi     2
              rss        25
npr           gdelt      14
              newsapi     2
              rss        10
the_guardian  gdelt      10
              rss        46


## 2. Data Coverage: Articles per Outlet per Month

Before looking at drift, I need to understand the data density. A month with 1-2 articles
will produce a very noisy proportion estimate - a single article changing label flips the
proportion by 50-100 percentage points. This is a fundamental limitation of the dataset
and is worth showing explicitly.

In [10]:
# Monthly proportions - the core data structure for all drift analysis below
monthly = monthly_bias_proportions(df)

# Pivot the article counts into a coverage table: rows=months, cols=outlets
coverage = monthly.pivot_table(
    index="month", columns="outlet", values="n_articles", fill_value=0
)

# Reorder columns to a logical left-right ordering
col_order = [c for c in ["bbc", "npr", "fox_news", "the_guardian"] if c in coverage.columns]
coverage = coverage[col_order]

print("Articles per outlet per month")
print()
print(coverage.to_string())

Articles per outlet per month

outlet    bbc   npr  fox_news  the_guardian
month                                      
2025-11   3.0   1.0       4.0           5.0
2025-12   3.0   4.0       4.0           0.0
2026-01   1.0   2.0       2.0           3.0
2026-02   7.0   2.0       2.0           0.0
2026-03   6.0   2.0       0.0           2.0
2026-04   3.0   3.0       1.0           0.0
2026-05  38.0  12.0      27.0          46.0


## 3. Monthly Bias Proportions Table

The full proportions table for all four outlets. Left/centre/right percentages add to 100
for each row. Months with very few articles (n=1 or 2) will show extreme proportions - this
is expected noise, not a signal.

In [11]:
for outlet in DRIFT_OUTLETS:
    outlet_df = monthly[monthly["outlet"] == outlet].copy()
    if outlet_df.empty:
        continue
    print(f"--- {outlet} ---")
    print(outlet_df[["month", "n_articles", "left_pct", "centre_pct", "right_pct"]].to_string(index=False))
    print()

--- bbc ---
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           3      66.7        33.3        0.0
2025-12           3      33.3        66.7        0.0
2026-01           1       0.0       100.0        0.0
2026-02           7      14.3        85.7        0.0
2026-03           6       0.0       100.0        0.0
2026-04           3       0.0       100.0        0.0
2026-05          38      36.8        57.9        5.3

--- npr ---
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           1     100.0         0.0        0.0
2025-12           4     100.0         0.0        0.0
2026-01           2     100.0         0.0        0.0
2026-02           2     100.0         0.0        0.0
2026-03           2     100.0         0.0        0.0
2026-04           3     100.0         0.0        0.0
2026-05          12     100.0         0.0        0.0

--- fox_news ---
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           4      25.0         0.0       75.0
202

## 4. Baseline Calculation

The baseline is the outlet's "normal" bias distribution - calculated from its first two months
of data (Nov + Dec 2025 for outlets with GDELT data starting in November). All later months
are compared against this baseline to detect drift.

Two months is a short baseline window, which means the baseline itself is noisy. In a production
system you would want at least 4-6 months of data before establishing a baseline. With the
available data, this is the best I can do - documented here rather than hidden.

In [12]:
# Compute baselines using first 2 months of data per outlet
baselines = compute_baselines(monthly, n_months=2)

print("Baselines (first 2 months per outlet)")
print()
for outlet, b in baselines.items():
    print(f"{outlet}")
    print(f"  Baseline months: {b['baseline_months']}")
    print(f"  left={b['left_mean']:.1f}%  centre={b['centre_mean']:.1f}%  right={b['right_mean']:.1f}%")
    print()

Baselines (first 2 months per outlet)

bbc
  Baseline months: ['2025-11', '2025-12']
  left=50.0%  centre=50.0%  right=0.0%

fox_news
  Baseline months: ['2025-11', '2025-12']
  left=37.5%  centre=0.0%  right=62.5%

npr
  Baseline months: ['2025-11', '2025-12']
  left=100.0%  centre=0.0%  right=0.0%

the_guardian
  Baseline months: ['2025-11', '2026-01']
  left=56.6%  centre=43.4%  right=0.0%



## 5. Drift Events

A drift event is flagged when a month's label proportion deviates from the baseline by more
than 20 percentage points. For example: if BBC's baseline left% is 50% and a later month
shows 10% left, that is a 40pp deviation - flagged as drift.

20pp is a large absolute shift - chosen because the small monthly sample sizes make
standard-deviation-based thresholds unreliable. A 20pp shift in a month with 5 articles
means 1 article changed label - which is not necessarily meaningful. But a 20pp shift in a
month with 40 articles (May 2026) is more robust.

In [13]:
drift_events = detect_drift_events(monthly, baselines, threshold_pct=20.0)

print(f"Drift events detected: {len(drift_events)}")
print()
if not drift_events.empty:
    print(drift_events.to_string(index=False))

Drift events detected: 6

      outlet   month  label  baseline_pct  observed_pct  deviation
         bbc 2026-02   left          50.0          14.3       35.7
         bbc 2026-02 centre          50.0          85.7       35.7
         bbc 2026-03   left          50.0           0.0       50.0
         bbc 2026-03 centre          50.0         100.0       50.0
the_guardian 2026-05   left          56.6          21.7       35.0
the_guardian 2026-05 centre          43.4          69.6       26.2


## 6. Time Series Charts

One subplot per outlet. Each line shows how the proportion of a bias label changes month
over month. The dashed horizontal lines show the baseline mean for each label. Vertical
dotted markers flag drift events.

Marker size is proportional to the number of articles in that month - larger markers mean
more data and more reliable estimates. Small markers should be read with caution.

In [14]:
# Colours consistent with the rest of the project
LABEL_COLOURS = {"left": "#4e79a7", "centre": "#76b7b2", "right": "#e15759"}

# Display names for the subplots
OUTLET_LABELS = {
    "bbc":         "BBC",
    "npr":         "NPR",
    "fox_news":    "Fox News",
    "the_guardian": "The Guardian",
}


def make_drift_chart(monthly: pd.DataFrame, baselines: dict, drift_events: pd.DataFrame) -> go.Figure:
    """
    2x2 grid of time series charts, one per outlet.
    Each subplot shows left/centre/right proportions over time, with baseline markers.
    """
    outlets = DRIFT_OUTLETS
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[OUTLET_LABELS.get(o, o) for o in outlets],
        shared_yaxes=False,
        vertical_spacing=0.15,
        horizontal_spacing=0.1,
    )

    # Track which labels have already been added to the legend
    # so we don't get duplicate legend entries across subplots
    legend_shown = set()

    for idx, outlet in enumerate(outlets):
        row = idx // 2 + 1
        col = idx % 2 + 1

        outlet_df = monthly[monthly["outlet"] == outlet].sort_values("month")
        if outlet_df.empty:
            continue

        months = outlet_df["month"].tolist()

        # Scale marker size to n_articles - clamped between 6 and 18
        max_n = outlet_df["n_articles"].max()
        marker_sizes = [
            6 + int(12 * (n / max_n)) for n in outlet_df["n_articles"]
        ]

        for label in ["left", "centre", "right"]:
            colour = LABEL_COLOURS[label]
            show_legend = label not in legend_shown

            fig.add_trace(
                go.Scatter(
                    x=months,
                    y=outlet_df[f"{label}_pct"].tolist(),
                    mode="lines+markers",
                    name=label,
                    line=dict(color=colour, width=2),
                    marker=dict(color=colour, size=marker_sizes),
                    showlegend=show_legend,
                    legendgroup=label,
                    # Show n_articles on hover so the reader can assess reliability
                    customdata=outlet_df["n_articles"].tolist(),
                    hovertemplate=(
                        f"{label}: %{{y:.1f}}%<br>n=%{{customdata}}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )
            legend_shown.add(label)

            # Baseline dashed horizontal line
            if outlet in baselines:
                base_val = baselines[outlet][f"{label}_mean"]
                fig.add_trace(
                    go.Scatter(
                        x=[months[0], months[-1]],
                        y=[base_val, base_val],
                        mode="lines",
                        line=dict(color=colour, width=1, dash="dash"),
                        showlegend=False,
                        hoverinfo="skip",
                    ),
                    row=row, col=col,
                )

        # Vertical dotted lines for drift events for this outlet
        outlet_events = drift_events[drift_events["outlet"] == outlet]
        flagged_months = outlet_events["month"].unique()

        for flagged_month in flagged_months:
            fig.add_vline(
                x=flagged_month,
                line_dash="dot",
                line_color="orange",
                line_width=2,
                row=row, col=col,
            )

        fig.update_yaxes(range=[0, 105], title_text="% articles", row=row, col=col)
        fig.update_xaxes(tickangle=45, row=row, col=col)

    fig.update_layout(
        title="Bias Label Distribution Over Time (dashed = baseline, orange dotted = drift event)",
        height=650,
        width=950,
        legend_title="Bias label",
    )
    return fig


make_drift_chart(monthly, baselines, drift_events)

## 7. Drift Event Summary Table

A readable summary of which outlets flagged drift, in which month, and how large the deviation was.

In [15]:
if drift_events.empty:
    print("No drift events detected at the 20pp threshold.")
else:
    # Join article counts so the reader can assess how reliable each flagged month is
    n_articles_map = monthly.set_index(["outlet", "month"])["n_articles"].to_dict()
    drift_events["n_articles"] = drift_events.apply(
        lambda r: n_articles_map.get((r["outlet"], r["month"]), 0), axis=1
    )
    print(drift_events.to_string(index=False))

      outlet   month  label  baseline_pct  observed_pct  deviation  n_articles
         bbc 2026-02   left          50.0          14.3       35.7           7
         bbc 2026-02 centre          50.0          85.7       35.7           7
         bbc 2026-03   left          50.0           0.0       50.0           6
         bbc 2026-03 centre          50.0         100.0       50.0           6
the_guardian 2026-05   left          56.6          21.7       35.0          46
the_guardian 2026-05 centre          43.4          69.6       26.2          46


## 8. Per-Outlet Interpretation

A quick summary of what the time series shows for each outlet - written after looking at the
actual numbers rather than before. This is the interpretive step.

In [16]:
print("=== BBC ===")
bbc = monthly[monthly["outlet"] == "bbc"].sort_values("month")
print(bbc[["month", "n_articles", "left_pct", "centre_pct", "right_pct"]].to_string(index=False))
print()

print("=== NPR ===")
npr = monthly[monthly["outlet"] == "npr"].sort_values("month")
print(npr[["month", "n_articles", "left_pct", "centre_pct", "right_pct"]].to_string(index=False))
print()

print("=== Fox News ===")
fox = monthly[monthly["outlet"] == "fox_news"].sort_values("month")
print(fox[["month", "n_articles", "left_pct", "centre_pct", "right_pct"]].to_string(index=False))
print()

print("=== The Guardian ===")
guardian = monthly[monthly["outlet"] == "the_guardian"].sort_values("month")
print(guardian[["month", "n_articles", "left_pct", "centre_pct", "right_pct"]].to_string(index=False))

=== BBC ===
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           3      66.7        33.3        0.0
2025-12           3      33.3        66.7        0.0
2026-01           1       0.0       100.0        0.0
2026-02           7      14.3        85.7        0.0
2026-03           6       0.0       100.0        0.0
2026-04           3       0.0       100.0        0.0
2026-05          38      36.8        57.9        5.3

=== NPR ===
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           1     100.0         0.0        0.0
2025-12           4     100.0         0.0        0.0
2026-01           2     100.0         0.0        0.0
2026-02           2     100.0         0.0        0.0
2026-03           2     100.0         0.0        0.0
2026-04           3     100.0         0.0        0.0
2026-05          12     100.0         0.0        0.0

=== Fox News ===
  month  n_articles  left_pct  centre_pct  right_pct
2025-11           4      25.0         0.0       75.0
202

## 9. Observations

**On data density**: the GDELT historical window gives 1-7 articles per outlet per month.
At that density, a single article changing predicted label shifts the monthly proportion by
14-50 percentage points. The drift signals visible in the charts are real changes in the
label distribution, but they cannot be cleanly separated from sampling noise. The May 2026
live data (10-46 articles) is far more reliable - that month's proportions can be read with
much more confidence.

**On the 7-day rolling average**: the original drift plan specified a 7-day rolling average
of `bias_confidence` by label, derived from daily data. The GDELT data is sampled at monthly
granularity, not daily. Monthly proportions are used here as a pragmatic substitute. To
implement the 7-day rolling average properly, I would need to run the live ingestion pipeline
daily for several weeks - possible in a production system, not available in this prototype.

**NPR**: classified as 100% left across every single month. This is almost certainly a
classifier artefact rather than a real signal. NPR's editorial style is calm and policy-focused,
which the AllSides training data likely maps consistently to the left label. The classifier
never predicts centre or right for NPR, which suggests overconfident labeling rather than
genuine one-sided reporting. No drift is detectable here because there is nothing to deviate from.

**BBC**: the most interesting outlet in the dataset. The GDELT historical months show a mix
of left and centre labels, while the May 2026 live data (40 articles) shows a clearer picture.
Any drift flags on BBC months with n=1-3 articles should be read with caution.

**Fox News**: the GDELT historical data classifies roughly half of Fox News articles as left,
which is counterintuitive. This reflects a known limitation of the model: the AllSides training
data is US-centric and the left/right labels are derived from US political framing. Fox News
articles about international events (Ukraine, Iran) may not exhibit the same language patterns
as the AllSides training set, leading to misclassification. The May 2026 live data (27 articles)
shows a more expected distribution of 67% right.

**The Guardian**: historically classified as predominantly left (80% in Nov 2025), shifting
toward centre in the live data (70% centre in May 2026). This is the clearest apparent drift
signal in the dataset - though the small GDELT sample sizes make it hard to call this a real
editorial shift vs. topic variation across ingestion windows.

**What this shows for a portfolio context**: the drift monitoring pipeline works end to end -
data ingestion, date parsing across two different formats, monthly aggregation, baseline
computation, and drift event detection. The honest limitation is data volume: meaningful drift
monitoring requires sustained daily ingestion over weeks or months. GDELT provides the historical
depth but not the density. This is the kind of tradeoff that comes up in every production ML
monitoring system.